In [27]:
import json
import mlflow
import onnxmltools
from lightgbm import early_stopping

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from onnxmltools.convert.common.data_types import FloatTensorType

import sys
sys.path.append("..")
from utils import get_table, set_mlflow_experiment
from feature_engineering import time_based_split, infer_column_types, bool_to_int, datetime_to_int64, get_feature_names_from_preprocessor, get_feature_names_from_preprocessor

In [30]:
# Naming
experiment_name: str = "lightgbm_pipeline"
run_name: str = "test_1"

# Model training
train_frac: float = 0.8
val_frac: float = 0.1
drop_importance_below: float = 0.0  # es. 0.0 = niente drop, oppure 1e-6 / 0.0001
onnx_export_path: str = "artifacts/model.onnx"


features_cols = ["chance1x2_quote_diffRealCurr1", "chance1x2_quote_diffRealCurr2", "evaluation_val1x2", "evaluation_valScala", "evaluation_valMetrica", "chance1x2_bookkeeping_status", "chance1x2_quote_diffInitialCurr2", "chance1x2_quote_diffInitialCurr1", "chance1x2_quote_current1"]
target_col = "win_1"

# Default LGBM params (puoi modificarli)
lgbm_params = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 64,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "objective": "binary",
    "metric": "auc",
}

# lgbm_params = {
#     "n_estimators": 500,
#     "learning_rate": 0.03,
#
#     "num_leaves": 16,
#     "max_depth": 4,
#     "min_data_in_leaf": 200,
#
#     "subsample": 0.7,
#     "subsample_freq": 1,
#     "colsample_bytree": 0.7,
#
#     "reg_lambda": 5.0,
#     "reg_alpha": 1.0,
#
#     "objective": "binary",
#     "metric": "auc",      # IMPORTANTISSIMO
#     "random_state": 42,
#     "n_jobs": -1,
# }

# Load Data

In [6]:
set_mlflow_experiment(experiment_name=experiment_name)

query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

In [9]:
type(df_loaded['time'].iloc[0]) == pd.Timestamp

True

# Data Processing

## Train/Test/Val Split

In [25]:
unuseful_cols = ['team_league', 'team_home', 'team_away']

df = df_loaded.copy()

df = df.drop(unuseful_cols, axis=1)
# Parse time col
# df["time"] = pd.to_datetime(df["time"], errors="coerce")

# Basic sanity
df = df.dropna(how='all', axis=1)
# target must be 0/1
df[target_col] = df[target_col].astype(int)

num_cols, bool_cols, datetime_cols, cat_cols = infer_column_types(df, target_col)

train_df, val_df, test_df = time_based_split(df=df, time_col="time", train_frac=train_frac, val_frac=val_frac)

X_train = train_df[features_cols]
y_train = train_df[target_col].values

X_val = val_df[features_cols]
y_val = val_df[target_col].values

X_test = test_df[features_cols]
y_test = test_df[target_col].values

tot_cols = features_cols + [target_col]
num_cols = [x for x in num_cols if x in tot_cols]
bool_cols = [x for x in bool_cols if x in tot_cols]
datetime_cols = [x for x in datetime_cols if x in tot_cols]
cat_cols = [x for x in cat_cols if x in tot_cols]

# Preprocess
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("astype", FunctionTransformer(lambda x: x.astype("float64"), validate=False)),
            ]), num_cols),
        ("bool", Pipeline(steps=[
            ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=bool_cols), validate=False)),
            ("cast", FunctionTransformer(bool_to_int, validate=False)),
        ]), bool_cols),
        ("dt", Pipeline(steps=[
            ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=datetime_cols), validate=False)),
            ("cast", FunctionTransformer(datetime_to_int64, validate=False)),
        ]), datetime_cols),
        # ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=False,
)


model = LGBMClassifier(**lgbm_params)

# pipeline sklearn
pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", model),
])


In [43]:
with mlflow.start_run(run_name=run_name) as mlflow_run:
    # Log split info
    mlflow.log_params({
        "train_frac": train_frac,
        "val_frac": val_frac,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        **{f"lgbm__{k}": v for k, v in lgbm_params.items()},
    })

    # Fit with early stopping using validation
    # NB: early_stopping via fit params (LightGBM sklearn API)
    pipe.fit(
        X_train, y_train,
        clf__eval_set=[(preprocessor.fit_transform(X_val), y_val)],  # val transformed
        clf__eval_metric="auc",
        clf__callbacks=[early_stopping(stopping_rounds=50, verbose=False)],    )

    # Predict proba
    p_train = pipe.predict_proba(X_train)[:, 1]
    p_val = pipe.predict_proba(X_val)[:, 1]
    p_test = pipe.predict_proba(X_test)[:, 1]

    pred_train = pipe.predict(X_train)
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)


    auc_train = roc_auc_score(y_train, p_train) if len(np.unique(y_train)) > 1 else np.nan
    auc_val = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan
    auc_test = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else np.nan

    precision_train = precision_score(y_train, pred_train)
    precision_val = precision_score(y_val, pred_val)
    precision_test = precision_score(y_test, pred_test)

    mlflow.log_metrics({
        "auc_train": float(auc_train) if np.isfinite(auc_train) else -1.0,
        "auc_val": float(auc_val) if np.isfinite(auc_val) else -1.0,
        "auc_test": float(auc_test) if np.isfinite(auc_test) else -1.0,
        "precision_train": float(precision_train) if np.isfinite(precision_train) else -1.0,
        "precision_val": float(precision_val) if np.isfinite(precision_val) else -1.0,
        "precision_test": float(precision_test) if np.isfinite(precision_test) else -1.0,
    })

    # Feature importance
    prep_fitted = pipe.named_steps["prep"]
    feature_names = get_feature_names_from_preprocessor(prep_fitted)
    

    booster = pipe.named_steps["clf"].booster_
    importances = booster.feature_importance(importance_type="gain")
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "importance_gain": importances
    }).sort_values("importance_gain", ascending=False)

    imp_csv = "feature_importance_gain.csv"
    imp_df.to_csv(imp_csv, index=False)
    mlflow.log_artifact(imp_csv)

    # Optional drop features under threshold & retrain
    if drop_importance_below > 0.0:
        keep_mask = imp_df["importance_gain"].values > drop_importance_below
        kept_features = imp_df.loc[keep_mask, "feature"].tolist()
        dropped = int((~keep_mask).sum())
        mlflow.log_params({
            "drop_importance_below": drop_importance_below,
            "dropped_features_count": dropped,
            "kept_features_count": len(kept_features),
        })

        # Per droppare in modo robusto con one-hot: selezioniamo colonne DOPO il preprocessor
        # Strategy: trasformiamo X_* e poi addestriamo un secondo LGBM su matrice ridotta.
        Xtr = prep_fitted.transform(X_train)
        Xva = prep_fitted.transform(X_val)
        Xte = prep_fitted.transform(X_test)

        keep_idx = np.where(keep_mask)[0]
        Xtr_k = Xtr[:, keep_idx]
        Xva_k = Xva[:, keep_idx]
        Xte_k = Xte[:, keep_idx]

        model2 = LGBMClassifier(**lgbm_params)
        model2.fit(
            Xtr_k, y_train,
            eval_set=[(Xva_k, y_val)],
            eval_metric="auc",
        )

        p_val2 = model2.predict_proba(Xva_k)[:, 1]
        p_test2 = model2.predict_proba(Xte_k)[:, 1]
        auc_val2 = roc_auc_score(y_val, p_val2) if len(np.unique(y_val)) > 1 else np.nan
        auc_test2 = roc_auc_score(y_test, p_test2) if len(np.unique(y_test)) > 1 else np.nan

        mlflow.log_metrics({
            "auc_val_dropped": float(auc_val2) if np.isfinite(auc_val2) else -1.0,
            "auc_test_dropped": float(auc_test2) if np.isfinite(auc_test2) else -1.0,
        })

        # Log modello ridotto come artifact “secondario”
        mlflow.lightgbm.log_model(model2, name="lgbm_model_retrained_after_drop")
        # Salviamo anche gli indici keep per riprodurre a runtime
        with open("artifacts/kept_feature_indices.json", "w") as f:
            json.dump(keep_idx.tolist(), f)
        mlflow.log_artifact("artifacts/kept_feature_indices.json")

    # Log modello pipeline (preprocess + lgbm)
    mlflow.sklearn.log_model(pipe, name="sklearn_pipeline_lgbm", input_example=X_train.dropna().iloc[:1])

    # ------------------------------------
    # ONNX export (modello puro LightGBM)
    # ------------------------------------
    # Per ONNX più compatto: esportiamo il Booster e a runtime replichi il preprocessing.
    # Qui esportiamo il modello *addestrato sullo spazio trasformato*:
    Xtr_trans = prep_fitted.transform(X_train)
    n_features_trans = Xtr_trans.shape[1]


    # Convert LightGBM booster to ONNX
    # NOTE: output probabilità: dipende dal converter; spesso produce label+probabilities
    initial_types = [("input", FloatTensorType([None, n_features_trans]))]
    onnx_model = onnxmltools.convert_lightgbm(
        booster,
        initial_types=initial_types,
        target_opset=15,
    )

    with open(onnx_export_path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    mlflow.log_artifact(onnx_export_path)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Number of positive: 1817, number of negative: 2375
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000515 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1150
[LightGBM] [Info] Number of data points in the train set: 4192, number of used features: 9
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.433445 -> initscore=-0.267811
[LightGBM] [Info] Start training from score -0.267811
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [44]:
pipe.named_steps["clf"].best_iteration_

60

# Calculate Treshold and EV

In [49]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

from sklearn.metrics import precision_score
from sklearn.frozen import FrozenEstimator

odds_name = "chance1x2_quote_current1"

def get_threshold_and_test_ev(
    val_df, test_df,
    model, odds_name, target_col,
    P_MIN=0.75, N_MIN_PERC=0.1,
    EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.7, weight_nbets=0.3
):
    assert abs(weight_ev + weight_nbets - 1.0) < 1e-6

    X_val = val_df.drop(target_col, axis=1)
    y_val = val_df[target_col]
    X_test = test_df.drop(target_col, axis=1)
    y_test = test_df[target_col]


    # -----------------------------
    # 2. Calibration (fit on VAL)
    # -----------------------------
    calibrator = CalibratedClassifierCV(
        estimator=FrozenEstimator(model),
        method="isotonic",
        cv=5   # qui cv non serve per fare fit del model, ma per calibrare in CV (consigliato)
    )
    calibrator.fit(X_val, y_val)

    # -----------------------------
    # 3. Predict on VAL
    # -----------------------------
    p_val = calibrator.predict_proba(X_val)[:, 1]


    val_df["p_cal"] = p_val
    val_df["odds"] = val_df[odds_name]
    val_df["EV"] = val_df["p_cal"] * val_df["odds"] - 1

    # -----------------------------
    # 4. Sweep thresholds (VAL)
    # -----------------------------
    results = []
    THRESHOLDS = np.linspace(0.55, 0.90, 71)
    N_MIN = int(len(y_val) * N_MIN_PERC)

    for t in THRESHOLDS:
        sel = val_df[val_df["p_cal"] >= t]
        if len(sel) < N_MIN:
            continue

        precision = precision_score(
            sel[target_col].astype(int),
            np.ones(len(sel))
        )
        mean_ev = sel["EV"].mean()

        if precision < P_MIN or not (EV_MIN <= mean_ev <= EV_MAX):
            continue

        results.append({
            "threshold": t,
            "mean_EV_val": mean_ev,
            "precision_val": precision,
            "n_bets_val": len(sel),
            "n_bets_val_perc": int(len(sel) / len(y_val) * 100),
        })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        print("Nessuna soglia valida trovata")
        return None

    # -----------------------------
    # 5. Score robusto (VAL)
    # -----------------------------
    EV_CENTER = (EV_MIN + EV_MAX) / 2

    res_df["score"] = (
        weight_nbets * (res_df["n_bets_val"] / res_df["n_bets_val"].max()) +
        weight_ev * (
            1 - abs(res_df["mean_EV_val"] - EV_CENTER) / (EV_MAX - EV_MIN)
        )
    )

    optimal = res_df.sort_values("score", ascending=False).iloc[0]
    t_star = optimal["threshold"]

    # -----------------------------
    # 6. FINAL EVALUATION ON TEST
    # -----------------------------
    p_test = calibrator.predict_proba(X_test)[:, 1]

    test_df["p_cal"] = p_test
    test_df["odds"] = test_df[odds_name]
    test_df["EV"] = test_df["p_cal"] * test_df["odds"] - 1

    sel_test = test_df[test_df["p_cal"] >= t_star]

    test_metrics = {
        "threshold": t_star,
        "n_bets_test_perc": int(len(sel_test) / len(y_test) * 100),
        "mean_EV_test": sel_test["EV"].mean() if len(sel_test) > 0 else np.nan,
        "ROI_test": sel_test["EV"].sum() / len(sel_test) if len(sel_test) > 0 else np.nan,
        "precision_test": precision_score(
            sel_test[target_col].astype(int),
            np.ones(len(sel_test))
        ) if len(sel_test) > 0 else np.nan
    }

    return {
        "val": optimal.to_dict(),
        "test": test_metrics
    }


optimal = get_threshold_and_test_ev(
    val_df, test_df, pipe, odds_name, target_col,
    P_MIN=0.75, EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.9, weight_nbets=0.1, N_MIN_PERC=0.01
)
print(optimal)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
{'val': {'threshold': 0.7550000000000001, 'mean_EV_val': 0.10868686868686867, 'precision_val': 0.8888888888888888, 'n_bets_val': 18.

In [84]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score
from sklearn.base import clone

def select_threshold_oof(
    val_df, model, odds_name, target_col,
    P_MIN=0.75, N_MIN_PERC=0.01,
    EV_MIN=0.05, EV_MAX=0.5,
    weight_ev=0.9, weight_nbets=0.1,
    cal_method="sigmoid",
    n_splits=5, random_state=42
):
    X_val = val_df.drop(columns=[target_col])
    y_val = val_df[target_col].astype(int).values
    odds = val_df[odds_name].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    p_oof = np.zeros(len(val_df), dtype=float)

    # 1) OOF calibrated probabilities (NO leakage per soglia)
    for tr_idx, va_idx in skf.split(X_val, y_val):
        base = clone(model)
        # qui assumo che "model" sia già un estimator completo (pipeline compresa) che fa fit
        base.fit(X_val.iloc[tr_idx], y_val[tr_idx])

        cal = CalibratedClassifierCV(estimator=base, method=cal_method, cv="prefit")
        cal.fit(X_val.iloc[va_idx], y_val[va_idx])

        p_oof[va_idx] = cal.predict_proba(X_val.iloc[va_idx])[:, 1]

    EV = p_oof * odds - 1

    THRESHOLDS = np.linspace(0.55, 0.90, 71)
    N_MIN = max(1, int(len(y_val) * N_MIN_PERC))

    rows = []
    for t in THRESHOLDS:
        mask = p_oof >= t
        n = mask.sum()
        if n < N_MIN:
            continue

        prec = y_val[mask].mean()  # equivalente a precision_score(y_true, ones)
        mean_ev = EV[mask].mean()

        if prec < P_MIN or not (EV_MIN <= mean_ev <= EV_MAX):
            continue

        rows.append({
            "threshold": float(t),
            "mean_EV_val": float(mean_ev),
            "precision_val": float(prec),
            "n_bets_val": int(n),
            "n_bets_val_perc": int(n / len(y_val) * 100),
        })

    res_df = pd.DataFrame(rows)
    if res_df.empty:
        return None, None

    EV_CENTER = (EV_MIN + EV_MAX) / 2
    res_df["score"] = (
        weight_nbets * (res_df["n_bets_val"] / res_df["n_bets_val"].max()) +
        weight_ev * (1 - np.abs(res_df["mean_EV_val"] - EV_CENTER) / (EV_MAX - EV_MIN))
    )

    best = res_df.sort_values("score", ascending=False).iloc[0]
    return best.to_dict(), p_oof

def fit_calibrator_on_full_val_and_test(
    val_df, test_df, model, odds_name, target_col, t_star,
    cal_method="sigmoid", cv=5
):
    X_val = val_df.drop(columns=[target_col])
    y_val = val_df[target_col].astype(int).values
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col].astype(int).values

    # Fit base model su tutto val
    base = clone(model).fit(X_val, y_val)

    # Calibrazione “vera” sul val (puoi anche fare cv=5)
    cal = CalibratedClassifierCV(estimator=base, method=cal_method, cv=cv)
    cal.fit(X_val, y_val)

    p_test = cal.predict_proba(X_test)[:, 1]
    EV_test = p_test * test_df[odds_name].values - 1
    mask = p_test >= t_star

    sel_n = mask.sum()
    return {
        "threshold": float(t_star),
        "n_bets_test": int(sel_n),
        "n_bets_test_perc": int(sel_n / len(y_test) * 100),
        "mean_EV_test": float(np.mean(EV_test[mask])) if sel_n else np.nan,
        "precision_test": float(np.mean(y_test[mask])) if sel_n else np.nan,
    }

optimal = fit_calibrator_on_full_val_and_test(
    val_df, test_df, model, odds_name, target_col, t_star,
    cal_method="sigmoid", cv=5
)
print(optimal)

NameError: name 't_star' is not defined